# CSE5280 — Ray Tracing Engine (Extended)
**Author:** *Your Name Here*  
**Course:** CSE5280 — Computer Graphics  
**Date:** Spring 2026

---

## Table of Contents
1. [Imports & Setup](#setup)
2. [Ray Equation & Class](#ray)
3. [Material Model](#material)
4. [Sphere — Geometry & Intersection](#sphere)
5. [Plane — Geometry & Intersection](#plane)
6. [Camera & Coordinate Conversion](#camera)
7. [Phong Reflection Model & Lighting](#phong)
8. [Shadows (Hard & Soft)](#shadows)
9. [Reflection & Refraction (Snell's Law)](#reflrefr)
10. [Recursive Ray Tracer](#tracer)
11. [Feature Flags](#flags)
12. [Scene Assembly](#scene)
13. [Render & Results](#render)
14. [Multiple Viewpoints](#views)
15. [Debugging Tips](#debug)

---


In [ ]:
# ── 1. Imports & Setup ─────────────────────────────────────────────────────────
import numpy as np
from PIL import Image as im
import random, math


## 2. Ray Equation & Class <a name="ray"></a>

A **ray** is a half-line starting at the eye $\mathbf{e}$ and traveling in direction $\hat{\mathbf{d}}$:

$$
\mathbf{p}(t) = \mathbf{e} + \hat{\mathbf{d}}\,t, \qquad t \geq 0
$$

where $\hat{\mathbf{d}} = (\mathbf{s} - \mathbf{e}) / \|\mathbf{s} - \mathbf{e}\|$ is the **unit** direction from the eye to the image-plane sample point $\mathbf{s}$.  
The parameter $t$ measures distance along the ray; larger $t$ = farther from the eye.


In [ ]:
# ── 2. Ray class ───────────────────────────────────────────────────────────────
class Ray:
    """Parametric ray: p(t) = e + d*t  (d is a unit vector).

    Parameters
    ----------
    e         : array-like (3,) – ray origin
    d         : array-like (3,) – ray direction (normalized internally if normalize=True)
    normalize : bool            – set False when direction is already unit-length
    """
    def __init__(self, e, d, normalize=True):
        self.e = np.asarray(e, dtype=float)
        self.d = np.asarray(d, dtype=float)
        if normalize:
            n = np.linalg.norm(self.d)
            if n > 1e-10:
                self.d = self.d / n

    def at(self, t):
        """3-D point at parameter t: p = e + d*t."""
        return self.e + self.d * t


## 3. Material Model <a name="material"></a>

A `Material` stores every optical property a surface needs.

| Attribute | Symbol | Description |
|-----------|--------|-------------|
| `color` | $\mathbf{k}$ | Base RGB colour in $[0,1]^3$ |
| `ambient` | $k_a$ | Fraction of global ambient absorbed |
| `diffuse` | $k_d$ | Lambertian diffuse coefficient |
| `specular` | $k_s$ | Specular highlight coefficient |
| `shininess` | $n_s$ | Phong exponent (larger = tighter highlight) |
| `reflectivity` | $k_r$ | Mirror-like reflectance in $[0,1]$ |
| `transparency` | $k_t$ | Transmittance in $[0,1]$ (0 = opaque) |
| `ior` | $\eta$ | Index of refraction (1.0 = air, ≈1.5 = glass) |
| `glossiness` | $\sigma$ | Std-dev of glossy perturbation (0 = perfect mirror) |


In [ ]:
# ── 3. Material class ──────────────────────────────────────────────────────────
class Material:
    """All optical properties of a surface.

    Colors are stored in [0, 1] (not [0, 255]).
    """
    def __init__(self, color,
                 ambient=0.1, diffuse=0.7, specular=0.3, shininess=32,
                 reflectivity=0.0, transparency=0.0, ior=1.0, glossiness=0.0):
        self.color        = np.asarray(color, dtype=float)
        self.ambient      = float(ambient)
        self.diffuse      = float(diffuse)
        self.specular     = float(specular)
        self.shininess    = float(shininess)
        self.reflectivity = float(reflectivity)
        self.transparency = float(transparency)
        self.ior          = float(ior)
        self.glossiness   = float(glossiness)


## 4. Sphere — Geometry & Intersection <a name="sphere"></a>

The implicit equation of a sphere with centre $\mathbf{c}$ and radius $r$:

$$
\|\mathbf{p} - \mathbf{c}\|^2 = r^2
$$

Substituting the ray $\mathbf{p}(t) = \mathbf{e} + \hat{\mathbf{d}}\,t$ and expanding yields:

$$
\underbrace{(\hat{\mathbf{d}}\cdot\hat{\mathbf{d}})}_{A}\,t^2
  + \underbrace{2\,\hat{\mathbf{d}}\cdot(\mathbf{e}-\mathbf{c})}_{B}\,t
  + \underbrace{(\mathbf{e}-\mathbf{c})\cdot(\mathbf{e}-\mathbf{c}) - r^2}_{C}
  = 0
$$

Discriminant $\Delta = B^2 - 4AC$:

- $\Delta < 0$ → **miss**  
- $\Delta \geq 0$ → two roots $t_{1,2}$; take the smallest **positive** root

We add a small $\varepsilon = 10^{-4}$ to $t_{\min}$ to avoid **self-intersection** ("shadow acne"), a common artefact where the surface blocks its own shadow ray.

The **outward unit normal** at point $\mathbf{p}$ on the sphere:

$$\hat{\mathbf{n}} = \frac{\mathbf{p} - \mathbf{c}}{r}$$


In [ ]:
# ── 4. Sphere class ────────────────────────────────────────────────────────────
class Sphere:
    """Sphere defined by centre, radius, and a Material.

    Parameters
    ----------
    center   : (3,) array-like
    radius   : float
    material : Material
    """
    def __init__(self, center, radius, material):
        self.center   = np.asarray(center, dtype=float)
        self.radius   = float(radius)
        self.material = material

    def intersect(self, ray):
        """Return smallest positive t (ray parameter) at intersection, or np.inf on miss."""
        oc   = ray.e - self.center
        A    = np.dot(ray.d, ray.d)          # = 1 if d is normalised
        B    = 2.0 * np.dot(ray.d, oc)
        C    = np.dot(oc, oc) - self.radius ** 2
        disc = B * B - 4.0 * A * C
        if disc < 0:
            return np.inf
        sq   = np.sqrt(disc)
        t1   = (-B - sq) / (2.0 * A)
        t2   = (-B + sq) / (2.0 * A)
        eps  = 1e-4                          # avoid self-intersection
        if t1 > eps: return t1
        if t2 > eps: return t2
        return np.inf

    def normal_at(self, p):
        """Outward unit normal at surface point p."""
        return (p - self.center) / self.radius


## 5. Plane — Geometry & Intersection <a name="plane"></a>

A plane through point $\mathbf{p}_1$ with outward unit normal $\hat{\mathbf{n}}$:

$$
(\mathbf{p} - \mathbf{p}_1) \cdot \hat{\mathbf{n}} = 0
$$

Substituting the ray and solving for $t$:

$$
t = \frac{(\mathbf{p}_1 - \mathbf{e}) \cdot \hat{\mathbf{n}}}{\hat{\mathbf{d}} \cdot \hat{\mathbf{n}}}
$$

The intersection is **invalid** when:
- $|\hat{\mathbf{d}} \cdot \hat{\mathbf{n}}| < \varepsilon$ → ray nearly parallel to the plane  
- $t \leq 0$ → intersection is behind the eye

#### Checkerboard Texture
The hit point is projected onto two world-space axes (here $x$ and $z$) and tiled:

$$
\text{tile}(p) = \left\lfloor p_x / s \right\rfloor + \left\lfloor p_z / s \right\rfloor \pmod{2}
$$

where $s$ is the tile scale in world units.


In [ ]:
# ── 5. Plane class ─────────────────────────────────────────────────────────────
class Plane:
    """Infinite plane defined by a point, a normal, and one or two Materials.

    Parameters
    ----------
    p1         : (3,) point on the plane
    n          : (3,) surface normal (need not be unit; normalised internally)
    material   : Material – primary colour / tile A
    checker    : bool     – enable checkerboard texture
    mat2       : Material – tile B (only used when checker=True)
    tile_scale : float    – world-space size of each checker square
    """
    def __init__(self, p1, n, material, checker=False, mat2=None, tile_scale=100.0):
        self.p1         = np.asarray(p1, dtype=float)
        self.n          = np.asarray(n,  dtype=float)
        self.n          = self.n / np.linalg.norm(self.n)
        self.material   = material
        self.checker    = checker
        self.mat2       = mat2 if mat2 is not None else material
        self.tile_scale = float(tile_scale)

    def intersect(self, ray):
        """Return t > 0 on hit, np.inf otherwise."""
        denom = np.dot(ray.d, self.n)
        if abs(denom) < 1e-8:          # Ray (nearly) parallel to plane
            return np.inf
        t = np.dot(self.p1 - ray.e, self.n) / denom
        return t if t > 1e-4 else np.inf

    def normal_at(self, p):
        """Planes have a constant normal everywhere."""
        return self.n.copy()

    def material_at(self, p):
        """Return Material at point p (supports checker pattern)."""
        if not self.checker:
            return self.material
        ix = int(math.floor(p[0] / self.tile_scale))
        iz = int(math.floor(p[2] / self.tile_scale))
        return self.material if (ix + iz) % 2 == 0 else self.mat2


## 6. Camera & Coordinate Conversion <a name="camera"></a>

The virtual pinhole camera sits at eye position $\mathbf{e}$ and has an image plane at focal distance $f$ directly in front.

Pixel $(i,j)$ maps to image-plane coordinates $(u,v)$ via:

$$
u = \left(j + \tfrac{1}{2}\right) - \frac{n_{\text{cols}}}{2}, \qquad
v = -\left(i + \tfrac{1}{2}\right) + \frac{n_{\text{rows}}}{2}
$$

(The $-$ sign on $v$ flips the vertical axis so that row 0 corresponds to the top of the image.)

The image-plane sample in 3-D is $\mathbf{s} = (u,\; v,\; e_z - f)^T$ (directly in front of the eye along $-z$).  
The primary ray direction is then $\hat{\mathbf{d}} = \text{normalize}(\mathbf{s} - \mathbf{e})$.

To render the **same scene from a different viewpoint** simply construct a new `Camera` with a different `eye` value.


In [ ]:
# ── 6. Camera class ────────────────────────────────────────────────────────────
class Camera:
    """Pinhole camera.

    Parameters
    ----------
    f      : float        focal distance (pixels)
    nrows  : int          image height in pixels
    ncols  : int          image width  in pixels
    eye    : (3,) or None eye / camera centre in world space (default: origin)
    """
    nchannels = 3

    def __init__(self, f, nrows, ncols, eye=None):
        self.f     = float(f)
        self.nrows = int(nrows)
        self.ncols = int(ncols)
        self.eye   = np.array(eye if eye is not None else [0.0, 0.0, 0.0], dtype=float)
        self.I     = np.zeros([self.nrows, self.ncols, self.nchannels])

    def make_ray(self, i, j):
        """Construct a primary ray through pixel (i, j)."""
        u = (j + 0.5) - self.ncols / 2.0
        v = -(i + 0.5) + self.nrows / 2.0
        # Image plane is at z = eye_z - f (directly in front)
        s = np.array([self.eye[0] + u,
                      self.eye[1] + v,
                      self.eye[2] - self.f])
        return Ray(self.eye, s - self.eye, normalize=True)


## 7. Phong Reflection Model & Lighting <a name="phong"></a>

The Phong model decomposes local illumination at a surface point into three components:

$$
\boxed{
I(\mathbf{p}) = \underbrace{k_a\,I_a}_{\text{ambient}}
              + \underbrace{k_d\,(\hat{\mathbf{L}}\cdot\hat{\mathbf{n}})}_{\text{diffuse}}
              + \underbrace{k_s\,(\hat{\mathbf{R}}\cdot\hat{\mathbf{v}})^{n_s}}_{\text{specular}}
}
$$

Each term is computed **per light source** and **per colour channel** (R, G, B independently).

| Vector | Definition |
|--------|-----------|
| $\hat{\mathbf{L}}$ | Unit vector from hit point toward the light |
| $\hat{\mathbf{n}}$ | Outward unit surface normal |
| $\hat{\mathbf{v}}$ | Unit vector from hit point toward the eye |
| $\hat{\mathbf{R}}$ | Reflection of $-\hat{\mathbf{L}}$ about $\hat{\mathbf{n}}$ |

The reflection direction is:

$$
\hat{\mathbf{R}} = 2\,(\hat{\mathbf{L}}\cdot\hat{\mathbf{n}})\,\hat{\mathbf{n}} - \hat{\mathbf{L}}
$$

Note: diffuse is clamped to $\max(0, \hat{\mathbf{L}}\cdot\hat{\mathbf{n}})$ to avoid negative light.  
Note: specular is similarly clamped.


## 8. Shadows — Hard & Soft <a name="shadows"></a>

### Hard Shadows (point light)

For each lit surface point $\mathbf{p}$, cast a **shadow ray** from $\mathbf{p}$ toward the light $\mathbf{L}$.  
If any scene object blocks the ray at $0 < t < \|\mathbf{L} - \mathbf{p}\|$, the point is in shadow:

$$
\text{shadow\_factor} = \begin{cases} 0 & \text{(blocked)} \\ 1 & \text{(unblocked)} \end{cases}
$$

### Soft Shadows (area light)

An **area light** is approximated by sampling $N_s$ random positions $\mathbf{L}_k$ on a disk of radius $r_{\text{area}}$ around the nominal light position.  
Each sample independently tests for blocking:

$$
\text{shadow\_factor} = \frac{1}{N_s}\sum_{k=1}^{N_s} \mathbf{1}\bigl[\text{shadow ray}_k \text{ unblocked}\bigr]
$$

Larger $N_s$ and larger $r_{\text{area}}$ produce smoother penumbra regions.


In [ ]:
# ── 7 & 8. PointLight, HitRecord, Scene ──────────────────────────────────────

class PointLight:
    """Point light with optional area (disk) for soft shadows.

    Parameters
    ----------
    position    : (3,) world position of the nominal light centre
    color       : (3,) light colour in [0, 1]
    intensity   : float overall brightness multiplier
    area_radius : float > 0 → soft shadows (disk area light); 0 → hard shadows
    """
    def __init__(self, position, color=None, intensity=1.0, area_radius=0.0):
        self.position    = np.asarray(position, dtype=float)
        self.color       = np.asarray(color if color is not None else [1,1,1], dtype=float)
        self.intensity   = float(intensity)
        self.area_radius = float(area_radius)

    def sample_position(self):
        """Return a (possibly jittered) point on this light source."""
        if self.area_radius < 1e-6:
            return self.position.copy()
        theta  = random.uniform(0.0, 2.0 * math.pi)
        r      = self.area_radius * math.sqrt(random.random())
        return self.position + np.array([r * math.cos(theta),
                                          0.0,
                                          r * math.sin(theta)])


class HitRecord:
    """All information produced by a ray–object intersection.

    Attributes
    ----------
    t        : float      ray parameter at hit
    point    : (3,)       world-space hit point
    normal   : (3,)       outward unit normal at hit
    material : Material   surface material at hit point
    obj      : object     the intersected geometric object
    """
    def __init__(self, t, point, normal, material, obj):
        self.t        = t
        self.point    = point
        self.normal   = normal
        self.material = material
        self.obj      = obj


class Scene:
    """Container for all scene geometry and lights.

    Usage:
        scene = Scene()
        scene.add_object(Sphere(...))
        scene.add_light(PointLight(...))
    """
    def __init__(self):
        self.objects = []
        self.lights  = []

    def add_object(self, obj):
        self.objects.append(obj)

    def add_light(self, light):
        self.lights.append(light)

    # ── Intersection ─────────────────────────────────────────────────────────

    def closest_hit(self, ray, t_min=1e-4, t_max=np.inf):
        """Return the HitRecord for the closest intersection in [t_min, t_max], or None."""
        best_t   = t_max
        best_hit = None
        for obj in self.objects:
            t = obj.intersect(ray)
            if t_min < t < best_t:
                best_t   = t
                p        = ray.at(t)
                n        = obj.normal_at(p)
                # Planes can return different materials for checker tiles
                mat = (obj.material_at(p)
                       if hasattr(obj, 'material_at')
                       else obj.material)
                best_hit = HitRecord(t, p, n, mat, obj)
        return best_hit

    # ── Shadow helpers ────────────────────────────────────────────────────────

    def _blocked(self, origin, light_pos):
        """True if the segment origin→light_pos is blocked by any object."""
        to_light = light_pos - origin
        dist     = np.linalg.norm(to_light)
        if dist < 1e-8:
            return False
        shadow_ray = Ray(origin, to_light, normalize=True)
        hit = self.closest_hit(shadow_ray, t_min=1e-4, t_max=dist - 1e-4)
        return hit is not None

    def shadow_factor(self, origin, light, n_samples=1):
        """Fraction of light samples visible from origin (1 = fully lit)."""
        lit = sum(0 if self._blocked(origin, light.sample_position()) else 1
                  for _ in range(n_samples))
        return lit / n_samples


## 9. Reflection & Refraction <a name="reflrefr"></a>

### 9.1 Perfect Mirror Reflection

Given incident unit direction $\hat{\mathbf{d}}$ (pointing **toward** the surface) and outward unit normal $\hat{\mathbf{n}}$:

$$
\hat{\mathbf{r}} = \hat{\mathbf{d}} - 2\,(\hat{\mathbf{d}}\cdot\hat{\mathbf{n}})\,\hat{\mathbf{n}}
$$

Since $\hat{\mathbf{d}}$ points toward the surface, $\hat{\mathbf{d}}\cdot\hat{\mathbf{n}} < 0$, making the reflected ray point away from the surface.

### 9.2 Glossy (Imperfect) Reflection

Glossy surfaces scatter reflected light within a small cone.  
Each glossy sample perturbs the mirror direction with Gaussian noise:

$$
\hat{\mathbf{r}}_{\text{glossy}} = \text{normalize}\!\left(\hat{\mathbf{r}} + \sigma\,\boldsymbol{\xi}\right),
\quad \boldsymbol{\xi} \sim \mathcal{N}(\mathbf{0}, \mathbf{I})
$$

Multiple samples are averaged to approximate the result of integrating over the BRDF lobe.

### 9.3 Snell's Law & Refraction

At an interface between media with indices of refraction $\eta_1$ and $\eta_2$:

$$
\eta_1 \sin\theta_1 = \eta_2 \sin\theta_2
$$

Vector form of the transmitted direction $\hat{\mathbf{t}}$:

$$
\hat{\mathbf{t}} = \frac{\eta_1}{\eta_2}\,\hat{\mathbf{d}}
   + \left(\frac{\eta_1}{\eta_2}\cos\theta_1 - \cos\theta_2\right)\hat{\mathbf{n}},
$$

where $\cos\theta_1 = -\hat{\mathbf{d}}\cdot\hat{\mathbf{n}}$ and $\cos\theta_2 = \sqrt{1 - \left(\frac{\eta_1}{\eta_2}\right)^2\sin^2\theta_1}$.

When the quantity under the square root is **negative**, **Total Internal Reflection** (TIR) occurs — the ray cannot exit the denser medium and is fully reflected instead.

**Entering vs. exiting:** we detect which side of the surface the ray is on via the sign of $\hat{\mathbf{d}}\cdot\hat{\mathbf{n}}$, and swap $\eta_1 \leftrightarrow \eta_2$ accordingly.


In [ ]:
# ── 9. Reflection and Refraction helpers ──────────────────────────────────────

def reflect(d, n):
    """Reflect unit direction d about unit normal n.

    d points *toward* the surface; returned direction points *away*.
    """
    return d - 2.0 * np.dot(d, n) * n


def refract(d, n, eta1, eta2):
    """Compute refracted direction using Snell's Law.

    Parameters
    ----------
    d    : unit incident direction (pointing toward the surface)
    n    : unit outward normal
    eta1 : IOR of the incident medium
    eta2 : IOR of the transmitted medium

    Returns
    -------
    unit transmitted direction, or None on Total Internal Reflection.
    """
    cos_theta1  = max(-1.0, min(1.0, -np.dot(d, n)))
    eta_ratio   = eta1 / eta2
    sin2_theta2 = eta_ratio ** 2 * (1.0 - cos_theta1 ** 2)

    if sin2_theta2 > 1.0:
        return None                  # Total Internal Reflection

    cos_theta2 = math.sqrt(max(0.0, 1.0 - sin2_theta2))
    t = eta_ratio * d + (eta_ratio * cos_theta1 - cos_theta2) * n
    norm_t = np.linalg.norm(t)
    return t / norm_t if norm_t > 1e-10 else None


## 11. Feature Flags <a name="flags"></a>

All major features can be switched on or off through a single `FEATURES` dictionary.  
This makes it easy to isolate individual effects, debug artefacts, or speed up test renders.


In [ ]:
# ── 10. Feature Flags ──────────────────────────────────────────────────────────
FEATURES = {
    # ── Lighting & Shadows ──────────────────────────────────────────────────
    'shadows'             : True,   # Cast shadow rays toward lights
    'soft_shadows'        : True,   # Area-light sampling (needs shadows=True)
    'soft_shadow_samples' : 8,      # Shadow rays per light (more = softer)

    # ── Recursive effects ───────────────────────────────────────────────────
    'reflection'          : True,   # Mirror / glossy reflections
    'refraction'          : True,   # Transparent objects via Snell's Law
    'glossy_samples'      : 4,      # Reflection rays per bounce (1 = sharp mirror)
    'max_depth'           : 4,      # Maximum recursion depth

    # ── Global illumination approximation ───────────────────────────────────
    'ambient_intensity'   : 0.20,   # Constant ambient light level [0, 1]
}


## 10. Recursive Ray Tracer <a name="tracer"></a>

The heart of the engine is `trace_ray`.  For each ray it:

1. Finds the **closest intersection** in the scene.
2. Returns the background colour if no object is hit.
3. Evaluates **Phong shading** from every light (with optional shadows).
4. If the surface is **reflective**, recursively traces one or more reflected rays and blends the result:
   $$
   I_{\text{final}} = (1 - k_r)\,I_{\text{Phong}} + k_r\,I_{\text{reflected}}
   $$
5. If the surface is **transparent**, traces a refracted ray and blends:
   $$
   I_{\text{final}} = (1 - k_t)\,I_{\text{current}} + k_t\,I_{\text{refracted}}
   $$
6. **Gamma correction** ($\gamma = 2.2$) is applied at the very end of the render loop to convert linear light values to display-ready pixel values.


In [ ]:
# ── 11. Recursive ray tracer ───────────────────────────────────────────────────

BACKGROUND_COLOR = np.array([0.05, 0.07, 0.12])   # Deep-blue sky

def phong_at_hit(hit, ray_d, eye, scene, features):
    """Phong shading for all lights at a given HitRecord.

    Returns linear RGB in [0, ~large].  Clamped later in trace_ray.

    Parameters
    ----------
    hit      : HitRecord
    ray_d    : unit ray direction (pointing *toward* surface)
    eye      : (3,) eye position
    scene    : Scene
    features : dict of feature flags
    """
    mat   = hit.material
    p     = hit.point
    n     = hit.normal
    v     = -ray_d                            # direction toward the eye

    # ── Ambient ───────────────────────────────────────────────────────────
    I = mat.ambient * features['ambient_intensity'] * mat.color

    # ── Per-light diffuse + specular ─────────────────────────────────────
    for light in scene.lights:
        # Shadow test
        if features['shadows']:
            n_samp = (features['soft_shadow_samples']
                      if features['soft_shadows'] else 1)
            sf = scene.shadow_factor(p, light, n_samp)
        else:
            sf = 1.0

        if sf < 1e-6:
            continue                          # Fully in shadow

        # Light vector (unit) and intensity (no distance falloff by design:
        # the scene uses large coordinates, and falloff would make everything
        # nearly black.  For a physically-based engine, replace with 1/dist^2.)
        to_light = light.position - p
        L        = to_light / np.linalg.norm(to_light)
        I_light  = light.color * light.intensity

        # Diffuse (Lambertian)
        diff = max(0.0, np.dot(L, n))
        I   += sf * mat.diffuse * diff * mat.color * I_light

        # Specular (Phong)
        R    = reflect(-L, n)
        spec = max(0.0, np.dot(R, v)) ** mat.shininess
        I   += sf * mat.specular * spec * I_light

    return I


def trace_ray(ray, scene, features, depth=0):
    """Recursively trace a ray and return RGB colour in [0, 1]^3.

    Parameters
    ----------
    ray      : Ray
    scene    : Scene
    features : dict of feature flags
    depth    : current recursion depth
    """
    # ── Depth limit (base case) ───────────────────────────────────────────
    if depth > features['max_depth']:
        return BACKGROUND_COLOR.copy()

    # ── Closest intersection ──────────────────────────────────────────────
    hit = scene.closest_hit(ray)
    if hit is None:
        return BACKGROUND_COLOR.copy()

    mat = hit.material
    p   = hit.point
    n   = hit.normal

    # ── Local Phong shading ───────────────────────────────────────────────
    color = phong_at_hit(hit, ray.d, ray.e, scene, features)

    # ── Reflection ────────────────────────────────────────────────────────
    if features['reflection'] and mat.reflectivity > 1e-4:
        refl_d  = reflect(ray.d, n)
        n_samp  = features['glossy_samples'] if mat.glossiness > 1e-6 else 1
        rc      = np.zeros(3)
        valid   = 0
        for _ in range(n_samp):
            jitter = (np.random.randn(3) * mat.glossiness
                      if mat.glossiness > 1e-6
                      else np.zeros(3))
            rd = refl_d + jitter
            rd_norm = np.linalg.norm(rd)
            if rd_norm < 1e-8:
                continue
            rc    += trace_ray(Ray(p, rd / rd_norm, normalize=False),
                               scene, features, depth + 1)
            valid += 1
        if valid > 0:
            rc /= valid
        color = (1.0 - mat.reflectivity) * color + mat.reflectivity * rc

    # ── Refraction ────────────────────────────────────────────────────────
    if features['refraction'] and mat.transparency > 1e-4:
        entering  = np.dot(ray.d, n) < 0
        eta1, eta2 = (1.0, mat.ior) if entering else (mat.ior, 1.0)
        n_face     = n if entering else -n
        td         = refract(ray.d, n_face, eta1, eta2)
        if td is not None:
            tc    = trace_ray(Ray(p, td, normalize=False),
                              scene, features, depth + 1)
            color = (1.0 - mat.transparency) * color + mat.transparency * tc

    return np.clip(color, 0.0, 1.0)


In [ ]:
# ── 12. Render function ────────────────────────────────────────────────────────

def render(camera, scene, features, gamma=2.2, verbose=True):
    """Render the scene and store the result in camera.I (float [0,255]).

    Parameters
    ----------
    camera   : Camera
    scene    : Scene
    features : dict of feature flags
    gamma    : float  gamma correction exponent (2.2 is standard for sRGB)
    verbose  : bool   print progress every 10% of rows

    Returns
    -------
    PIL Image (uint8 RGB)
    """
    nrows, ncols = camera.nrows, camera.ncols
    img = np.zeros((nrows, ncols, 3), dtype=float)

    milestone = max(1, nrows // 10)
    for i in range(nrows):
        if verbose and i % milestone == 0:
            print(f'  Rendering... {100 * i // nrows}%', end='\r')
        for j in range(ncols):
            ray   = camera.make_ray(i, j)
            color = trace_ray(ray, scene, features)
            img[i, j] = color

    # Gamma correction: linear → display
    img_gamma = np.power(np.clip(img, 0.0, 1.0), 1.0 / gamma)
    camera.I  = img_gamma * 255.0

    if verbose:
        print('  Rendering... 100% — done!          ')

    pil_img = im.fromarray(np.uint8(np.clip(camera.I, 0, 255)))
    display(pil_img)
    return pil_img


## 12. Scene Assembly <a name="scene"></a>

The default scene contains:

- A **checkerboard floor plane** (horizontal, $y = -80$)  
- A **red diffuse sphere** (back-left, mildly reflective)  
- A **green glossy sphere** (front-right, imperfect mirror)  
- A **glass sphere** at the centre demonstrating refraction + TIR  
- A **primary area light** (top-right, warm tone, soft shadows)  
- A **secondary fill light** (top-left, cool blue, no area — hard shadows)  


In [ ]:
# ── 13. Scene assembly ─────────────────────────────────────────────────────────

def build_default_scene():
    scene = Scene()

    # ── Materials ────────────────────────────────────────────────────────
    mat_red = Material(
        color=[0.85, 0.12, 0.10],
        ambient=0.08, diffuse=0.75, specular=0.35, shininess=50,
        reflectivity=0.08
    )
    mat_green = Material(
        color=[0.10, 0.78, 0.22],
        ambient=0.08, diffuse=0.55, specular=0.65, shininess=150,
        reflectivity=0.50, glossiness=0.035
    )
    mat_glass = Material(
        color=[0.92, 0.95, 1.00],
        ambient=0.03, diffuse=0.05, specular=0.95, shininess=300,
        reflectivity=0.12, transparency=0.80, ior=1.5
    )
    mat_floor_light = Material(
        color=[0.88, 0.88, 0.88],
        ambient=0.12, diffuse=0.70, specular=0.08, shininess=8,
        reflectivity=0.10
    )
    mat_floor_dark = Material(
        color=[0.12, 0.12, 0.12],
        ambient=0.08, diffuse=0.60, specular=0.03, shininess=4,
        reflectivity=0.06
    )

    # ── Geometry ─────────────────────────────────────────────────────────
    # Floor plane: y = -80, normal pointing up
    scene.add_object(Plane(
        p1=[0, -80, 0], n=[0, 1, 0],
        material=mat_floor_light,
        checker=True, mat2=mat_floor_dark, tile_scale=80.0
    ))

    # Red sphere — back left
    scene.add_object(Sphere(
        center=[-90,  0, -800], radius=80, material=mat_red
    ))

    # Green glossy sphere — front right
    scene.add_object(Sphere(
        center=[ 90,  0, -400], radius=80, material=mat_green
    ))

    # Glass sphere — centre
    scene.add_object(Sphere(
        center=[  0, 20, -600], radius=70, material=mat_glass
    ))

    # ── Lights ───────────────────────────────────────────────────────────
    # Main area light: warm, from top-right — produces soft shadows
    scene.add_light(PointLight(
        position=[250, 350, -100],
        color=[1.00, 0.92, 0.80],
        intensity=1.0,
        area_radius=70.0
    ))
    # Secondary fill light: cool blue, hard
    scene.add_light(PointLight(
        position=[-350, 200, -200],
        color=[0.45, 0.55, 0.80],
        intensity=0.40,
        area_radius=0.0
    ))

    return scene


## 13. Render — Default Scene <a name="render"></a>

We first render at **256 × 256** pixels for a quick preview.  
Increase to **512 × 512** or higher for a final-quality image.

> **Performance note:**  
> Soft shadows (`soft_shadow_samples`) and glossy reflections (`glossy_samples`) dominate render time.  
> Disable them via `FEATURES` for fast debug renders.


In [ ]:
# ── 14. Render — Default Scene ─────────────────────────────────────────────────
np.random.seed(42)

nrows, ncols = 256, 256    # ← change to 512 for a higher-quality render
focal        = 250.0

camera = Camera(focal, nrows, ncols, eye=[0, 30, 100])
scene  = build_default_scene()

print("Full render (all features ON):")
out = render(camera, scene, FEATURES)
out.save("render_full.png")
print("Saved → render_full.png")


## 14. Multiple Viewpoints <a name="views"></a>

Rendering from different eye positions requires only changing the `Camera`'s `eye` parameter.  
The scene geometry and lights stay identical.


In [ ]:
# ── 15. Multiple Viewpoints ────────────────────────────────────────────────────

viewpoints = {
    'Front'      : [  0,  30,  100],
    'High angle' : [  0, 350, -200],
    'Side'       : [450,  60, -500],
}

# Faster settings for the comparison grid
fast_feat = {**FEATURES,
             'soft_shadow_samples': 2,
             'glossy_samples'     : 1}

for name, eye in viewpoints.items():
    print(f"Viewpoint: {name}  eye={eye}")
    cam = Camera(focal, 128, 128, eye=eye)
    out = render(cam, scene, fast_feat, verbose=False)
    fname = f'render_{name.lower().replace(" ", "_")}.png'
    out.save(fname)
    display(out)
    print(f'  → Saved {fname}\n')


In [ ]:
# ── 16. Feature Toggle Comparison ──────────────────────────────────────────────

configs = {
    'All features ON'   : FEATURES,
    'No shadows'        : {**FEATURES, 'shadows':    False},
    'No reflection'     : {**FEATURES, 'reflection': False},
    'No refraction'     : {**FEATURES, 'refraction': False},
    'Hard shadows only' : {**FEATURES, 'soft_shadows': False, 'glossy_samples': 1},
}

cam_cmp = Camera(focal, 128, 128, eye=[0, 30, 100])

for label, feat in configs.items():
    print(f"{label}:")
    r = render(cam_cmp, scene, feat, verbose=False)
    r.save(f'compare_{label.lower().replace(" ", "_")}.png')
    display(r)
    print()


## 15. Debugging Tips <a name="debug"></a>

### Common Artefacts and Fixes

| Symptom | Likely Cause | Fix |
|---|---|---|
| **Black speckles** ("shadow acne") | Shadow ray hits own surface | Increase `t_min` epsilon (default `1e-4`) |
| **Completely black render** | All rays miss all objects, or lights too far | Print `t` values for a centre ray; check scene scale |
| **Extremely dark image** | Light too attenuated over distance | Reduce/remove distance falloff; raise `intensity` |
| **Flat spheres, no highlights** | Phong specular exponent too high or `v` wrong | Ensure `v = -ray.d` (toward eye); check `shininess` |
| **No refraction visible** | `transparency` or `ior` near zero | Set `transparency > 0`, `ior` ≠ 1.0 |
| **No soft shadows** | `area_radius = 0` on lights | Set `area_radius > 0` and `soft_shadows = True` |
| **Noisy glossy reflections** | Too few samples | Increase `glossy_samples` (4–16) |
| **Infinite recursion / slow** | Depth not limiting | Confirm `depth + 1` is passed recursively |
| **Checkerboard wrong size** | `tile_scale` off | Adjust `tile_scale` to match scene units |

### Debug Render Passes

```python
# ── Normal pass (maps surface normal [-1,1] → colour [0,1]) ──────────────────
# Great for spotting flipped normals (e.g., wrong-side plane)
def normal_pass(cam, scn):
    img = np.zeros((cam.nrows, cam.ncols, 3))
    for i in range(cam.nrows):
        for j in range(cam.ncols):
            hit = scn.closest_hit(cam.make_ray(i, j))
            if hit:
                img[i, j] = hit.normal * 0.5 + 0.5   # [-1,1] → [0,1]
    return im.fromarray(np.uint8(img * 255))

display(normal_pass(Camera(focal, 128, 128, eye=[0, 30, 100]), scene))
```

```python
# ── Depth pass (brighter = farther away) ────────────────────────────────────
def depth_pass(cam, scn, max_t=1200.0):
    img = np.zeros((cam.nrows, cam.ncols))
    for i in range(cam.nrows):
        for j in range(cam.ncols):
            hit = scn.closest_hit(cam.make_ray(i, j))
            if hit:
                img[i, j] = 1.0 - min(hit.t, max_t) / max_t
    return im.fromarray(np.uint8(img * 255))

display(depth_pass(Camera(focal, 128, 128, eye=[0, 30, 100]), scene))
```

```python
# ── Print hit info for centre pixel ─────────────────────────────────────────
cam_test = Camera(focal, 256, 256, eye=[0, 30, 100])
test_ray = cam_test.make_ray(128, 128)
hit = scene.closest_hit(test_ray)
if hit:
    print(f\"Hit: {type(hit.obj).__name__}, t={hit.t:.2f}, p={hit.point.round(1)}, n={hit.normal.round(3)}\")
else:
    print(\"No hit — background\")
```
